# Forex Trading Strategy Analysis

This notebook analyzes EUR/USD Forex trading data to identify profitable trading strategies based on various technical indicators and market conditions.

In [398]:
# Import required libraries
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import sys
import importlib

# Force reload of utils module to prevent caching
if 'utils' in sys.modules:
    del sys.modules['utils']

# Enable automatic reloading of modules
%load_ext autoreload
%autoreload 2

# Import utility functions (will be freshly loaded)
import utils
from utils import load_and_clean_data, calculate_profitable_trades, analyze_entry_timing, display_profitable_strategies

# Force reload utils to ensure latest version
importlib.reload(utils)

# Load the data
df = load_and_clean_data()

# Display profitable trades analysis
display(HTML("<h2>Profitable Trades</h2><p>What are simple trading ideas that are profitable?</p>"))
display(calculate_profitable_trades(df))
display(HTML("<p><b>Summary</b></p><ul><li>CH trades are out of question, they're not profitable.</li><li>3 pips extra is harming the strategy, while 1 pip can improve it slightly.</li><li>Overall any filtration is harmful.</ul>"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,Data,1:1 RRR,1:2 RRR,1:3 RRR
0,With Extra 1 pip,42.9%,25.5%,18.4%
1,With Extra 2 pips,42.9%,27.6%,14.3%
2,Total,38.8%,24.5%,20.4%
3,With Extra 3 pips,36.7%,22.4%,13.3%
4,Just BOS Trades,25.5%,17.3%,15.3%
5,With EMA Direction,23.5%,14.3%,11.2%
6,With EMA + BOS,18.4%,12.2%,10.2%
7,Against EMA Direction,15.3%,10.2%,9.2%
8,Just CH Trades,13.3%,7.1%,5.1%
9,With EMA + CH,5.1%,2.0%,1.0%


In [399]:
# Display entry timing analysis using the imported function
display(HTML("<h2>When To Enter</h2><p>Following market structure, price taps order block. This is when the signal is received. Now - when to enter the trade?</p>"))
df_entry_timing = analyze_entry_timing(df)
display(df_entry_timing)
display(HTML("<p><b>Summary</b></p><ul><li>Very interesting that here, my strategy with 1M confirmation candle is the worse.</li></ul>"))
display(HTML("<p><b>Open Questions</b></p><ul><li>That 5M Stop entry makes a lot of sense as all profitable strategies would have it, but in reality somehow it performs the worse. Why?</li><li>Why does 1M confirmation candle entry outperforms all other entries in the next setups analysis?</li></ul>"))

,Idea,Notation,Win Rate,With Extra,With Extra & 1:3 RRR
0,5M Stop,31W - 32L,49.2%,55.6%,28.6%
1,5M Breakout,30W - 35L,46.2%,52.3%,30.8%
2,5M Confirmation Candle,31W - 40L,43.7%,49.3%,28.2%
3,1M Confirmation Candle,38W - 60L,38.8%,42.9%,22.4%


In [400]:
# ================== STRATEGY FRAMEWORK ==================

def calculate_rrr_stats(data_df, strategy_name):
    """
    Calculate comprehensive Risk-Reward Ratio statistics for a trading strategy.
    
    This function evaluates a strategy's performance across different RRR targets
    and calculates key metrics including win rate, edge over breakeven, and
    expected outcome in R-multiples.
    
    Args:
        data_df (pd.DataFrame): Filtered DataFrame containing trades for this strategy
        strategy_name (str): Name of the strategy for labeling
    
    Returns:
        pd.DataFrame: Statistics table with the following metrics:
            - Total trades: Number of trades in the strategy
            - Wins/Losses: Count of profitable vs unprofitable trades
            - Win Rate: Percentage of winning trades
            - Edge: Win rate minus breakeven rate for the RRR
            - Outcome: Net result in R-multiples (profit factor)
    """
    total_trades = len(data_df)
    
    # RRR configurations with their breakeven win rates
    # For 1:1 RRR, you need 50% wins to break even
    # For 1:2 RRR, you need 33.3% wins to break even
    # For 1:3 RRR, you need 25% wins to break even
    rrr_configs = [
        (1, 50.0),   # 1:1 RRR
        (2, 33.3),   # 1:2 RRR
        (3, 25.0),   # 1:3 RRR
    ]
    
    # Handle empty strategy results
    if total_trades == 0:
        summary_data = {
            strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
        }
        for ratio, _ in rrr_configs:
            summary_data[f'1:{ratio} RRR'] = [0, 0, 0, '0.0%', '0.0%', '0R']
        return pd.DataFrame(summary_data)
    
    # Calculate statistics for each RRR level
    summary_data = {
        strategy_name: ['Total trades', 'Wins', 'Losses', 'Win Rate', 'Edge', 'Outcome']
    }
    
    for ratio, breakeven_rate in rrr_configs:
        # Find profitable trades for this RRR
        profitable = data_df[data_df['TP'] > ratio * data_df['SL']]
        wins = len(profitable)
        losses = total_trades - wins
        win_rate = (wins / total_trades) * 100
        
        # Edge is how much better we perform than breakeven
        edge = win_rate - breakeven_rate
        
        # Calculate expected outcome in R-multiples
        # Winners make 'ratio' R, losers lose 1R
        outcome = (wins * ratio) - losses
        
        summary_data[f'1:{ratio} RRR'] = [
            total_trades,
            wins,
            losses,
            f"{win_rate:.1f}%",
            f"{edge:.1f}%",
            f"{outcome}R"
        ]
    
    return pd.DataFrame(summary_data)


class Strategy:
    """
    Encapsulates a trading strategy with its filter logic and metadata.
    
    Attributes:
        name (str): Strategy identifier
        filter_func (callable): Function that filters trades based on strategy rules
        description (str): Human-readable description of the strategy
    """
    
    def __init__(self, name, filter_func, description=""):
        """
        Initialize a trading strategy.
        
        Args:
            name (str): Strategy name
            filter_func (callable): Lambda or function that takes df and returns filtered df
            description (str): Optional description of the strategy
        """
        self.name = name
        self.filter_func = filter_func
        self.description = description
    
    def apply(self, df):
        """
        Apply the strategy filter to a dataframe of trades.
        
        Args:
            df (pd.DataFrame): Trading data
            
        Returns:
            pd.DataFrame: Filtered trades matching strategy criteria
        """
        return self.filter_func(df)

In [401]:
# ================== STRATEGY DEFINITIONS ==================

# Initialize base strategy list
strategies = [
    Strategy(
        "Plain Strategy",
        lambda df: df,
        "Baseline: All trades without any filtering"
    )
]

In [402]:
# ================== STRATEGY FACTORY ==================

def create_strategy_library():
    """
    Create a comprehensive library of trading strategies for backtesting.
    
    This function generates strategies across multiple categories:
    1. Technical Indicators (EMA, BOS/CH)
    2. Risk Management (Stop Loss levels)
    3. Market Structure (Trend alignment)
    4. Time-based (Weekdays, News events)
    5. Combined filters (Multi-factor strategies)
    
    Returns:
        list: List of Strategy objects ready for backtesting
    """
    strategy_configs = []
    
    # ========== TECHNICAL INDICATOR STRATEGIES ==========
    # EMA (Exponential Moving Average) based strategies
    strategy_configs.extend([
        ("EMA + Trade Direction", lambda df: df[df['EMA'] == df['Direction']], "Trade with EMA trend"),
        ("EMA + Opposite Trade Direction", lambda df: df[df['EMA'] != df['Direction']], "Counter-trend trades"),
        ("EMA + BOS", lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS')], "Trend + Break of Structure"),
        ("EMA + CH", lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'CH')], "Trend + Change of Character"),
    ])
    
    # Market Structure strategies
    strategy_configs.extend([
        ("BOS", lambda df: df[df['BOS/CH'] == 'BOS'], "Break of Structure trades only"),
        ("CH", lambda df: df[df['BOS/CH'] == 'CH'], "Change of Character trades only"),
    ])
    
    # ========== RISK MANAGEMENT STRATEGIES ==========
    # Stop Loss size filtering
    strategy_configs.extend([
        ("Conservative: SL <= 2 pips", lambda df: df[df['SL'] <= 2], "Very tight stop losses"),
        ("Moderate Risk: SL 3-6 pips", lambda df: df[(df['SL'] >= 3) & (df['SL'] <= 6)], "Medium stop losses"),
        ("Aggressive: SL >= 7 pips", lambda df: df[df['SL'] >= 7], "Wide stop losses"),
    ])
    
    # Risk-adjusted market structure combinations
    strategy_configs.extend([
        ("BOS + Conservative SL", lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] <= 2)], "BOS with tight stops"),
        ("BOS + Moderate SL", lambda df: df[(df['BOS/CH'] == 'BOS') & (df['SL'] >= 3) & (df['SL'] <= 6)], "BOS with medium stops"),
    ])
    
    # ========== TIME-BASED STRATEGIES ==========
    # Day of week analysis
    # weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    # for day in weekdays:
    #     strategy_configs.append(
    #         (f"{day} Only", 
    #          lambda df, d=day: df[df['Weekday'] == d], 
    #          f"Trades on {day}")
    #     )
    
    # ========== HIGHER TIMEFRAME TREND ==========
    # 30-minute timeframe trend alignment
    strategy_configs.extend([
        ("With 30M Trend", lambda df: df[(df['30M Leg'].isin(['Above H', 'Above L']) & (df['Direction'] == 'Buy')) | (df['30M Leg'].isin(['Below H', 'Below L']) & (df['Direction'] == 'Sell'))], "Higher timeframe uptrend"),
    ])
    
    # ========== NEWS EVENT STRATEGIES ==========
    strategy_configs.extend([
        ("No News Events", lambda df: df[df['News Event'].isna()], "Avoid news volatility"),
        ("News Event Present", lambda df: df[~df['News Event'].isna()], "Trade during news"),
        ("News in 2+ Hours", lambda df: df[(~df['News Event'].isna()) & (df['Hours Until News'] >= 2)], "Trade before news impact"),
    ])
    
    # ========== COMBINED MULTI-FACTOR STRATEGIES ==========
    # These combine multiple indicators for potentially higher probability setups
    # strategy_configs.extend([
    #     ("EMA + BOS + Low Risk", lambda df: df[(df['EMA'] == df['Direction']) & (df['BOS/CH'] == 'BOS') & (df['SL'] <= 3)], "Triple confirmation setup"),
    #     ("BOS + Bullish 30M + No News", lambda df: df[(df['BOS/CH'] == 'BOS') & (df['30M Leg'].isin(['Above H', 'Above L'])) & (df['News Event'].isna())], "Clean trend continuation"),
    #     ("Strong Alignment + EMA", lambda df: df[((df['30M Leg'] == 'Above H') & (df['Direction'] == 'Buy') & (df['EMA'] == 'Buy')) | ((df['30M Leg'] == 'Below L') & (df['Direction'] == 'Sell') & (df['EMA'] == 'Sell'))], "Full trend confluence"),
    # ])
    
    # Convert configurations to Strategy objects
    return [Strategy(name, func, desc) for name, func, desc in strategy_configs]

# Add all strategies to the main list
strategies.extend(create_strategy_library())

# ================== STRATEGY EVALUATION ==================

def evaluate_all_strategies(df, strategies):
    """
    Run backtesting on all strategies and compile results.
    
    Args:
        df (pd.DataFrame): Trading data
        strategies (list): List of Strategy objects
        
    Returns:
        dict: Dictionary mapping strategy names to their performance DataFrames
    """
    strategy_results = {}
    
    for strategy in strategies:
        # Apply strategy filter
        filtered_df = strategy.apply(df)
        
        # Calculate RRR statistics
        summary_df = calculate_rrr_stats(filtered_df, strategy.name)
        
        # Store results
        strategy_results[strategy.name] = summary_df
    
    return strategy_results

def get_top_strategies(strategy_results, rrr_column, top_n=5):
    """
    Extract top performing strategies for a specific RRR.
    
    Args:
        strategy_results (dict): Dictionary of strategy results
        rrr_column (str): Column name for RRR (e.g., '1:2 RRR')
        top_n (int): Number of top strategies to return
        
    Returns:
        pd.DataFrame: Top strategies ranked by outcome
    """
    strategy_performance = []
    
    for strategy_name, df in strategy_results.items():
        # Extract performance metrics
        total_trades = df[rrr_column].iloc[0]
        wins = df[rrr_column].iloc[1]
        losses = df[rrr_column].iloc[2]
        win_rate = df[rrr_column].iloc[3]
        edge = df[rrr_column].iloc[4]
        outcome_str = df[rrr_column].iloc[5]
        
        # Parse outcome value for sorting
        outcome = int(outcome_str.replace('R', ''))
        
        strategy_performance.append({
            'Strategy': strategy_name,
            'Trades': total_trades,
            'Wins': wins,
            'Losses': losses,
            'Win Rate': win_rate,
            'Edge': edge,
            'Outcome': outcome_str,
            'outcome_value': outcome
        })
    
    # Sort by outcome and get top N
    top_strategies = sorted(
        strategy_performance, 
        key=lambda x: x['outcome_value'], 
        reverse=True
    )[:top_n]
    
    # Remove sorting key from display
    for strat in top_strategies:
        del strat['outcome_value']
    
    return pd.DataFrame(top_strategies)

# ================== RUN BACKTESTING ==================

# Evaluate all strategies
strategy_results = evaluate_all_strategies(df, strategies)

# Display top performing strategies for each RRR
display(HTML("<h2>🏆 TOP Performing Strategies</h2>"))

rrr_configs = [
    ('1:1 RRR', '1:1'),
    ('1:2 RRR', '1:2'),
    ('1:3 RRR', '1:3'),
]

for rrr_column, rrr_label in rrr_configs:
    display(HTML(f"<h3>Best {rrr_label} Strategies</h3>"))
    
    # Get and display top 5 strategies
    top_5_df = get_top_strategies(strategy_results, rrr_column, top_n=5)
    top_5_df = top_5_df.rename(columns={'Strategy': f'Top {rrr_label} Strategies'})
    
    # Style the table for better readability
    styled_df = top_5_df.style.set_properties(
        subset=[f'Top {rrr_label} Strategies'], 
        **{'width': '300px', 'font-weight': 'bold'}
    ).set_properties(
        subset=['Outcome'],
        **{'color': 'green', 'font-weight': 'bold'}
    )
    
    display(styled_df)
    print()  # Add spacing

,Top 1:1 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,33,28,54.1%,4.1%,5R
1,EMA + Opposite Trade Direction,38,21,17,55.3%,5.3%,4R
2,No News Events,54,29,25,53.7%,3.7%,4R
3,Aggressive: SL >= 7 pips,17,10,7,58.8%,8.8%,3R
4,Plain Strategy,98,50,48,51.0%,1.0%,2R


,Top 1:2 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,24,37,39.3%,6.0%,11R
1,News in 2+ Hours,26,12,14,46.2%,12.9%,10R
2,With 30M Trend,60,22,38,36.7%,3.4%,6R
3,EMA + Opposite Trade Direction,38,14,24,36.8%,3.5%,4R
4,BOS + Conservative SL,8,4,4,50.0%,16.7%,4R


,Top 1:3 Strategies,Trades,Wins,Losses,Win Rate,Edge,Outcome
0,BOS,61,22,39,36.1%,11.1%,27R
1,With 30M Trend,60,19,41,31.7%,6.7%,16R
2,Plain Strategy,98,28,70,28.6%,3.6%,14R
3,EMA + Opposite Trade Direction,38,13,25,34.2%,9.2%,14R
4,EMA + BOS,46,14,32,30.4%,5.4%,10R


In [403]:
# Display only profitable strategies by default
display_profitable_strategies(strategy_results)

,Plain Strategy,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,98,98,98
1,Wins,50,33,28
2,Losses,48,65,70
3,Win Rate,51.0%,33.7%,28.6%
4,Edge,1.0%,0.4%,3.6%
5,Outcome,2R,1R,14R


,EMA + Opposite Trade Direction,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,38,38,38
1,Wins,21,14,13
2,Losses,17,24,25
3,Win Rate,55.3%,36.8%,34.2%
4,Edge,5.3%,3.5%,9.2%
5,Outcome,4R,4R,14R


,BOS,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,61,61,61
1,Wins,33,24,22
2,Losses,28,37,39
3,Win Rate,54.1%,39.3%,36.1%
4,Edge,4.1%,6.0%,11.1%
5,Outcome,5R,11R,27R


,Moderate Risk: SL 3-6 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,53,53,53
1,Wins,27,16,13
2,Losses,26,37,40
3,Win Rate,50.9%,30.2%,24.5%
4,Edge,0.9%,-3.1%,-0.5%
5,Outcome,1R,-5R,-1R


,Aggressive: SL >= 7 pips,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,17,17,17
1,Wins,10,5,4
2,Losses,7,12,13
3,Win Rate,58.8%,29.4%,23.5%
4,Edge,8.8%,-3.9%,-1.5%
5,Outcome,3R,-2R,-1R


,BOS + Moderate SL,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,35,35,35
1,Wins,18,11,10
2,Losses,17,24,25
3,Win Rate,51.4%,31.4%,28.6%
4,Edge,1.4%,-1.9%,3.6%
5,Outcome,1R,-2R,5R


,With 30M Trend,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,60,60,60
1,Wins,31,22,19
2,Losses,29,38,41
3,Win Rate,51.7%,36.7%,31.7%
4,Edge,1.7%,3.4%,6.7%
5,Outcome,2R,6R,16R


,No News Events,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,54,54,54
1,Wins,29,18,16
2,Losses,25,36,38
3,Win Rate,53.7%,33.3%,29.6%
4,Edge,3.7%,0.0%,4.6%
5,Outcome,4R,0R,10R


,News in 2+ Hours,1:1 RRR,1:2 RRR,1:3 RRR
0,Total trades,26,26,26
1,Wins,14,12,9
2,Losses,12,14,17
3,Win Rate,53.8%,46.2%,34.6%
4,Edge,3.8%,12.9%,9.6%
5,Outcome,2R,10R,10R
